# Steam Games Dataset — Data Cleaning & Setup

This notebook loads the raw Steam Dataset 2025 (sourced from Kaggle), inspects
its structure and quality, and produces a cleaned dataset for downstream
SQL analysis and exploratory data analysis (EDA), which continue in
subsequent notebooks.

**Scope:** data loading, inspection, cleaning, and validation only.
No exploratory analysis or visualization of trends happens here.

**Source tables:** `applications` plus 9 relational lookup/junction tables
covering genres, categories, developers, publishers, and platforms.

**Contents:** 1 Setup · 2 Load · 3 Inspection · 4 Missing Values · 5 Duplicates · 6 Categorical · 7 Numerical · 8 Dates · 9 Relational Integrity · 10 Cleaning · 11 Validation · 12 Export · 13 Summary

## 1. Setup

In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_columns', None)  # show all columns, not truncated

con = duckdb.connect()
con.sql("INSTALL sqlite; LOAD sqlite;")
con.sql("CALL sqlite_attach('../data/steam_data.sqlite')")

┌─────────┐
│ Success │
│ boolean │
└─────────┘
  0 rows 

## 2. Load Raw Data

Loading all 10 relational tables from the SQLite database into DataFrames.

In [2]:
column_names = {
    'genres': ['id', 'name'],
    'categories': ['id', 'name'],
    'developers': ['id', 'name'],
    'publishers': ['id', 'name'],
    'platforms': ['id', 'name'],
    'application_genres': ['appid', 'genre_id'],
    'application_categories': ['appid', 'category_id'],
    'application_developers': ['appid', 'developer_id'],
    'application_publishers': ['appid', 'publisher_id'],
    'application_platforms': ['appid', 'platform_id'],
    'applications': [
        'appid', 'name', 'type', 'is_free', 'release_date', 'required_age',
        'short_description', 'supported_languages', 'header_image', 'background',
        'metacritic_score', 'recommendations_total', 'mat_supports_windows',
        'mat_supports_mac', 'mat_supports_linux', 'mat_initial_price', 'mat_final_price',
        'mat_discount_percent', 'mat_currency', 'mat_achievement_count', 'mat_pc_os_min',
        'mat_pc_processor_min', 'mat_pc_memory_min', 'mat_pc_graphics_min', 'mat_pc_os_rec',
        'mat_pc_processor_rec', 'mat_pc_memory_rec', 'mat_pc_graphics_rec',
        'created_at', 'updated_at'
    ],
}

# The CSV header row was imported as a data row in every table:
# load, drop that first row, and apply the real column names.
tables = {}
for name, cols in column_names.items():
    df = con.sql(f"SELECT * FROM {name}").df().iloc[1:].reset_index(drop=True)
    df.columns = cols
    tables[name] = df

genres = tables['genres']
categories = tables['categories']
developers = tables['developers']
publishers = tables['publishers']
platforms = tables['platforms']
application_genres = tables['application_genres']
application_categories = tables['application_categories']
application_developers = tables['application_developers']
application_publishers = tables['application_publishers']
application_platforms = tables['application_platforms']
applications = tables['applications']

for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

genres: 154 rows, 2 columns
categories: 462 rows, 2 columns
developers: 101,226 rows, 2 columns
publishers: 85,699 rows, 2 columns
platforms: 3 rows, 2 columns
application_genres: 587,515 rows, 2 columns
application_categories: 1,077,156 rows, 2 columns
application_developers: 250,817 rows, 2 columns
application_publishers: 223,048 rows, 2 columns
application_platforms: 334,671 rows, 2 columns
applications: 239,664 rows, 30 columns


In [3]:
tables = {
    "applications": applications, "genres": genres, "categories": categories,
    "developers": developers, "publishers": publishers, "platforms": platforms,
    "application_genres": application_genres, "application_categories": application_categories,
    "application_developers": application_developers, "application_publishers": application_publishers,
    "application_platforms": application_platforms,
}

for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

applications: 239,664 rows, 30 columns
genres: 154 rows, 2 columns
categories: 462 rows, 2 columns
developers: 101,226 rows, 2 columns
publishers: 85,699 rows, 2 columns
platforms: 3 rows, 2 columns
application_genres: 587,515 rows, 2 columns
application_categories: 1,077,156 rows, 2 columns
application_developers: 250,817 rows, 2 columns
application_publishers: 223,048 rows, 2 columns
application_platforms: 334,671 rows, 2 columns


## 3. Initial Inspection

Reviewing structure, data types, and sample records for each table before
identifying specific quality issues.

In [4]:
applications.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 239664 entries, 0 to 239663
Data columns (total 30 columns):
 #   Column                 Non-Null Count   Dtype 
---  ------                 --------------   ----- 
 0   appid                  239664 non-null  object
 1   name                   239664 non-null  object
 2   type                   236997 non-null  object
 3   is_free                239664 non-null  object
 4   release_date           202809 non-null  object
 5   required_age           239664 non-null  object
 6   short_description      224173 non-null  object
 7   supported_languages    222006 non-null  object
 8   header_image           239664 non-null  object
 9   background             239664 non-null  object
 10  metacritic_score       5299 non-null    object
 11  recommendations_total  22560 non-null   object
 12  mat_supports_windows   239664 non-null  object
 13  mat_supports_mac       239664 non-null  object
 14  mat_supports_linux     239664 non-null  object
 15  

In [5]:
applications.head()

,appid,name,type,is_free,release_date,required_age,short_description,supported_languages,header_image,background,metacritic_score,recommendations_total,mat_supports_windows,mat_supports_mac,mat_supports_linux,mat_initial_price,mat_final_price,mat_discount_percent,mat_currency,mat_achievement_count,mat_pc_os_min,mat_pc_processor_min,mat_pc_memory_min,mat_pc_graphics_min,mat_pc_os_rec,mat_pc_processor_rec,mat_pc_memory_rec,mat_pc_graphics_rec,created_at,updated_at
0,10,Counter-Strike,game,False,2000-11-01,0,Play the world's number 1 online action game. ...,"English<strong>*</strong>, French<strong>*</st...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,88.0,161854.0,True,True,True,999.0,999.0,0.0,USD,None,None,None,None,None,None,None,None,None,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
1,20,Team Fortress Classic,game,False,1999-04-01,0,One of the most popular online action games of...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,None,6633.0,True,True,True,499.0,499.0,0.0,USD,None,None,None,None,None,None,None,None,None,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
2,30,Day of Defeat,game,False,2003-05-01,0,Enlist in an intense brand of Axis vs. Allied ...,"English, French, German, Italian, Spanish - Spain",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,79.0,4308.0,True,True,True,499.0,499.0,0.0,USD,None,None,None,None,None,None,None,None,None,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
3,40,Deathmatch Classic,game,False,2001-06-01,0,Enjoy fast-paced multiplayer gaming with Death...,"English, French, German, Italian, Spanish - Sp...",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,None,2338.0,True,True,True,499.0,499.0,0.0,USD,None,None,None,None,None,None,None,None,None,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00
4,50,Half-Life: Opposing Force,game,False,1999-11-01,0,Return to the Black Mesa Research Facility as ...,"English, French, German, Korean",https://shared.akamai.steamstatic.com/store_it...,https://store.akamai.steamstatic.com/images/st...,None,22793.0,True,True,True,499.0,499.0,0.0,USD,None,None,None,None,None,None,None,None,None,2025-09-07 16:27:12.218587+00:00,2025-09-29 02:01:37.107239+00:00


In [6]:
applications.describe(include='all').T

,count,unique,top,freq
appid,239664,239664,3989830,1
name,239664,237760,Prism,6
type,236997,5,game,150279
is_free,239664,2,False,191390
release_date,202809,5496,2022-05-30,353
required_age,239664,21,0,235672
short_description,224173,206932,Embark on your adventure with player from all ...,284
supported_languages,222006,30503,English,54664
header_image,239664,239576,https://shared.akamai.steamstatic.com/store_it...,53
background,239664,239664,https://store.akamai.steamstatic.com/images/st...,1


In [7]:
for name, df in tables.items():
    if name != 'applications':
        print(f"--- {name} ---")
        print(df.dtypes)
        print(df.head(2))
        print()

--- genres ---
id      object
name    object
dtype: object
  id                        name
0  1                    Aventură
1  2  Многопользовательские игры

--- categories ---
id      object
name    object
dtype: object
  id                  name
0  1  Remote Play en móvil
1  2         Flera spelare

--- developers ---
id      object
name    object
dtype: object
  id                 name
0  1  Skala Entertainment
1  2            Flicksync

--- publishers ---
id      object
name    object
dtype: object
  id                   name
0  1    Skala Entertainment
1  2  Playdigious Originals

--- platforms ---
id      object
name    object
dtype: object
  id     name
0  1  windows
1  2      mac

--- application_genres ---
appid       object
genre_id    object
dtype: object
  appid genre_id
0    10      122
1    20      122

--- application_categories ---
appid          object
category_id    object
dtype: object
  appid category_id
0    10          72
1    10          90

--- application_deve

## 4. Missing Value Assessment

Quantifying missingness per column to decide how each should be handled.

In [8]:
missing = applications.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(applications) * 100).round(2)
missing_summary = pd.DataFrame({'missing_count': missing, 'missing_pct': missing_pct})
missing_summary[missing_summary['missing_count'] > 0]

,missing_count,missing_pct
metacritic_score,234365,97.79
recommendations_total,217104,90.59
mat_achievement_count,178263,74.38
mat_pc_graphics_rec,147465,61.53
mat_pc_memory_rec,142839,59.60
mat_pc_processor_rec,142547,59.48
mat_pc_os_rec,137705,57.46
mat_final_price,94741,39.53
mat_initial_price,94741,39.53
mat_discount_percent,94741,39.53


In [9]:
applications.groupby('is_free')['mat_initial_price'].apply(lambda x: x.isnull().mean() * 100)

is_free
False     24.278698
True     100.000000
Name: mat_initial_price, dtype: float64

In [10]:
print(genres.shape)
print(genres.sort_values('name').head(20))

(154, 2)
      id                  name
133  134             360 Video
38    39             Abenteuer
119  120                Acción
80    81            Accounting
8      9     Acesso Antecipado
144  145                 Actie
121  122                Action
5      6               Acțiune
112  113             Adventure
152  160                 Akció
21    22                 Akcja
135  136                  Aksi
13    14               Aksiyon
44    45  Animation & Modeling
67    68      Audio Production
53    54              Aventura
60    61              Aventure
0      1              Aventură
41    42             Avventura
12    13                Azione


In [11]:
print(categories.sort_values('name').head(30))

      id                                   name
7      8                   Achievement di Steam
36    37                           Achievementy
269  270          Additional High-Quality Audio
52    53                  Adjustable Difficulty
324  325                   Adjustable Text Size
227  228                         Aile Paylaşımı
451  492         Alkalmazáson belüli vásárlások
229  230                  Alternativas de cores
241  242  Benutzerdefinierte Lautstärkeregelung
70    71                Berbagi dengan Keluarga
40    41             Bölüm düzenleyicisi içerir
256  257                         Camera Comfort
426  427               Caméra et confort de vue
10    11                     Captions available
13    14             Cartas Colecionáveis Steam
100  101          Carte collezionabili di Steam
304  305                Cartes à échanger Steam
179  180              Cartonașe de schimb Steam
349  350                    Chat Speech-to-text
56    57                    Chat Text-to

In [12]:
pd.set_option('display.max_rows', None)
print(genres.sort_values('name').to_string())

      id                        name
133  134                   360 Video
38    39                   Abenteuer
119  120                      Acción
80    81                  Accounting
8      9           Acesso Antecipado
144  145                       Actie
121  122                      Action
5      6                     Acțiune
112  113                   Adventure
152  160                       Akció
21    22                       Akcja
135  136                        Aksi
13    14                     Aksiyon
44    45        Animation & Modeling
67    68            Audio Production
53    54                    Aventura
60    61                    Aventure
0      1                    Aventură
41    42                   Avventura
12    13                      Azione
63    64                        Ação
28    29               Basit Eğlence
23    24            Bağımsız Yapımcı
37    38                     Bildung
31    32                      Casual
65    66                     Corrida
7

### 4.1 Consolidating Genre Translations

The `genres` table contains 154 rows, but many represent the same genre
translated into different languages (e.g., "Action" appears separately as
Action, Acción, Actie, Aksiyon, Ação, and others). Below we map every
genre ID to a canonical English genre name and collapse `application_genres`
onto these canonical labels.

In [13]:
genre_id_to_canonical = {
     
    # Action
    '120': 'Action', '145': 'Action', '122': 'Action', '6': 'Action', '160': 'Action',
    '22': 'Action', '136': 'Action', '14': 'Action', '13': 'Action', '64': 'Action',
    '133': 'Action', '125': 'Action', '154': 'Action', '8': 'Action', '80': 'Action', '142': 'Action',

    # Adventure
    '39': 'Adventure', '54': 'Adventure', '61': 'Adventure', '1': 'Adventure', '42': 'Adventure',
    '113': 'Adventure', '18': 'Adventure', '132': 'Adventure', '26': 'Adventure', '108': 'Adventure',
    '57': 'Adventure', '101': 'Adventure', '43': 'Adventure', '128': 'Adventure', '150': 'Adventure',
    '94': 'Adventure', '85': 'Adventure',

    # Casual
    '32': 'Casual', '110': 'Casual', '34': 'Casual', '35': 'Casual', '92': 'Casual',
    '52': 'Casual', '29': 'Casual', '135': 'Casual', '127': 'Casual', '90': 'Casual',

    # Early Access
    '9': 'Early Access', '107': 'Early Access',

    # Education
    '38': 'Education', '130': 'Education',

    # Free to Play
    '44': 'Free to Play', '93': 'Free to Play', '28': 'Free to Play', '117': 'Free to Play',
    '95': 'Free to Play', '36': 'Free to Play', '21': 'Free to Play', '167': 'Free to Play',
    '75': 'Free to Play',

    # Game Development
    '123': 'Game Development', '86': 'Game Development',

    # Indie
    '31': 'Indie', '76': 'Indie', '3': 'Indie', '126': 'Indie', '24': 'Indie',
    '96': 'Indie', '146': 'Indie', '115': 'Indie', '7': 'Indie', '72': 'Indie', '48': 'Indie',

    # Massively Multiplayer
    '19': 'Massively Multiplayer', '16': 'Massively Multiplayer', '55': 'Massively Multiplayer',
    '59': 'Massively Multiplayer', '25': 'Massively Multiplayer', '148': 'Massively Multiplayer',
    '58': 'Massively Multiplayer', '2': 'Massively Multiplayer',

    # RPG
    '144': 'RPG', '65': 'RPG', '51': 'RPG', '40': 'RPG', '119': 'RPG', '69': 'RPG', '124': 'RPG',

    # Racing
    '66': 'Racing', '73': 'Racing', '15': 'Racing', '74': 'Racing', '63': 'Racing',
    '121': 'Racing', '149': 'Racing',

    # Simulation
    '10': 'Simulation', '17': 'Simulation', '114': 'Simulation', '91': 'Simulation', '11': 'Simulation',
    '141': 'Simulation', '67': 'Simulation', '143': 'Simulation', '37': 'Simulation', '109': 'Simulation',
    '138': 'Simulation', '137': 'Simulation', '99': 'Simulation', '147': 'Simulation', '20': 'Simulation',
    '103': 'Simulation', '41': 'Simulation', '71': 'Simulation', '87': 'Simulation',

    # Sports
    '129': 'Sports', '77': 'Sports', '89': 'Sports', '60': 'Sports', '111': 'Sports',
    '104': 'Sports', '62': 'Sports', '140': 'Sports', '97': 'Sports',

    # Strategy
    '139': 'Strategy', '46': 'Strategy', '102': 'Strategy', '118': 'Strategy', '78': 'Strategy',
    '4': 'Strategy', '5': 'Strategy', '88': 'Strategy', '70': 'Strategy', '30': 'Strategy',
    '56': 'Strategy', '84': 'Strategy', '105': 'Strategy', '112': 'Strategy',

    # Distinct standalone genres (kept as their own category, not translations)
    '134': '360 Video',
    '81': 'Accounting',
    '45': 'Animation & Modeling',
    '68': 'Audio Production',
    '47': 'Design & Illustration',
    '33': 'Documentary',
    '23': 'Episodic',
    '79': 'Gore',
    '131': 'Movie',
    '49': 'Nudity',
    '53': 'Photo Editing',
    '12': 'Sexual Content',
    '151': 'Short',
    '50': 'Software Training',
    '82': 'Tutorial',
    '116': 'Utilities',
    '83': 'Video Production',
    '106': 'Violent',
    '27': 'Web Publishing', '98': 'Web Publishing',

    # Unclear / not confidently identified
    '100': 'RYO', # Although i think its related to japanese

}

# Map application_genres onto canonical names
application_genres['genre_canonical'] = application_genres['genre_id'].map(genre_id_to_canonical)

# Check: did every genre_id get mapped? (should be 0 unmapped)
unmapped = application_genres[application_genres['genre_canonical'].isnull()]
print(f"Unmapped rows: {len(unmapped)}")
print(unmapped['genre_id'].unique())

Unmapped rows: 0
[]


In [14]:
print(f"Original unique genre IDs: {application_genres['genre_id'].nunique()}")
print(f"Canonical unique genres: {application_genres['genre_canonical'].nunique()}")
print(application_genres['genre_canonical'].value_counts())

Original unique genre IDs: 154
Canonical unique genres: 34
genre_canonical
Indie                    134192
Action                    84444
Casual                    77752
Adventure                 76028
Simulation                47945
RPG                       45935
Strategy                  43997
Free to Play              21823
Early Access              14731
Sports                     8601
Racing                     7541
Massively Multiplayer      6076
Design & Illustration      3688
Web Publishing             2976
Utilities                  2078
Education                  1788
Game Development           1731
Animation & Modeling       1141
Violent                     861
Video Production            741
Software Training           651
Audio Production            634
Gore                        533
Movie                       494
Photo Editing               401
Sexual Content              292
Nudity                      272
Documentary                  56
Accounting                   

In [15]:
dup = application_genres[application_genres.duplicated(subset=['appid', 'genre_canonical'], keep=False)]
dup.merge(genres, left_on='genre_id', right_on='id')[['appid', 'genre_id', 'name', 'genre_canonical']]

,appid,genre_id,name,genre_canonical
0,34330,5,Strategy,Strategy
1,34330,139,Estrategia,Strategy


In [16]:
print(application_genres.duplicated(subset=['appid', 'genre_canonical']).sum())

1


In [17]:
application_genres = (
    application_genres
    .drop_duplicates(subset=['appid', 'genre_canonical'])
    .drop(columns='genre_id', errors='ignore')
    .reset_index(drop=True)
)

### 4.2a Filtering Categories to English Labels

Like `genres`, the `categories` table stores the same concept under many languages
(e.g. "Steam Trading Cards" also appears as *Cartes à échanger Steam*,
*Cartas Colecionáveis Steam*, and others), so 462 IDs represent far fewer real categories.

**Approach:** Categories are a secondary dimension for this project, so instead of
building a full translation mapping (as done for genres), only the English-labelled
categories are kept:

- **Kept:** 61 English categories, after merging English-side duplicates that refer to the
  same feature under different labels or IDs (e.g. *Co-op online* / *Online co-op* →
  *Online Co-op*; *DLC* → *Downloadable Content*).
- **Validated the filter:** ranking the dropped categories by frequency showed three
  English labels (*PvP*, *MMO*, *LAN PvP*) had been missed in the first pass; these were
  added back. Every other dropped category is a translated label with under 100 rows.
- **Cost of the shortcut:** 0.35% of game–category rows were dropped, and 82 duplicate
  (game, category) pairs created by the merges were removed.

**Trade-off:** a game whose only recorded category was a non-English label loses that
tag. At 0.35% of rows this does not materially affect category-level analysis.

In [18]:
pd.set_option('display.max_rows', None)
print(categories.sort_values('name').to_string())

      id                                                      name
7      8                                      Achievement di Steam
36    37                                              Achievementy
269  270                             Additional High-Quality Audio
52    53                                     Adjustable Difficulty
324  325                                      Adjustable Text Size
227  228                                            Aile Paylaşımı
451  492                            Alkalmazáson belüli vásárlások
229  230                                     Alternativas de cores
241  242                     Benutzerdefinierte Lautstärkeregelung
70    71                                   Berbagi dengan Keluarga
40    41                                Bölüm düzenleyicisi içerir
256  257                                            Camera Comfort
426  427                                  Caméra et confort de vue
10    11                                        Captions avail

### 4.2b Filtering Categories to English Entries

Like `genres`, the `categories` table contains many locale-translated
duplicates. Given time constraints, categories are filtered to their
English-named entries rather than fully consolidated across all languages
(unlike genres, which were fully mapped since they are central to this
analysis). A small number of English-side near-duplicates (e.g.,
"Multi-player" vs "Multiplayer") are also merged.

In [19]:
category_ids_to_keep = [
    270, 53, 325, 257, 11, 350, 57, 160, 179, 211, 221, 72, 169, 125, 339,
    127, 159, 130, 75, 423, 326, 108, 218, 40, 193, 212, 292, 416, 45,
    133, 4, 403, 195, 380, 340, 113, 307, 286, 324, 383, 397, 192, 415,
    154, 347, 229, 436, 142, 124, 337, 68, 201, 283, 129, 85, 27, 119,
    175, 224, 10, 343, 77, 301, 67, 17, 128, 371,
    90, 121, 65,   # PvP, MMO, LAN PvP: English labels missed in the first pass
]

# Same concept under different English labels or duplicate IDs -> dominant label's ID
category_consolidation = {
    221: 380, 211: 380, 113: 380,   # Co-op online / Co-op Online / Online co-op -> Online Co-op
    179: 212,                        # Co-op LAN -> LAN Co-op
    127: 159,                        # DLC -> Downloadable Content
    133: 4,                          # Multi-player -> Multiplayer
    124: 142,                        # Singleplayer -> Single-player
    128: 17,                         # VR Supported -> VR Support
    383: 324,                        # duplicate Remote Play Together ID
}

categories_english = categories[categories['id'].astype(int).isin(category_ids_to_keep)].copy()
categories_english['id'] = categories_english['id'].astype(int)
categories_english = categories_english[~categories_english['id'].isin(list(category_consolidation))]

application_categories['category_id'] = (
    application_categories['category_id'].astype(int).replace(category_consolidation)
)

application_categories_clean = (
    application_categories[application_categories['category_id'].isin(categories_english['id'])]
    .merge(categories_english[['id', 'name']].rename(columns={'id': 'category_id', 'name': 'category_name'}),
           on='category_id', how='left')
    .drop_duplicates(subset=['appid', 'category_id'])
    .reset_index(drop=True)
)

dropped = application_categories[~application_categories['category_id'].isin(categories_english['id'])]

print(f"Distinct categories kept: {len(categories_english)}")
print(f"Rows: {len(application_categories):,} -> {len(application_categories_clean):,}")
print(f"Rows dropped as non-English: {len(dropped):,} ({len(dropped)/len(application_categories):.2%})")
print(application_categories_clean['category_name'].value_counts().head(8))
print(application_categories_clean['category_name'].value_counts().tail(6))

Distinct categories kept: 61
Rows: 1,077,156 -> 1,073,302
Rows dropped as non-English: 3,772 (0.35%)
category_name
Single-player                 200245
Family Sharing                156924
Steam Achievements             95440
Full controller support        60471
Steam Cloud                    56712
Downloadable Content           53406
Multiplayer                    51887
Partial Controller Support     32222
Name: count, dtype: int64
category_name
Includes Source SDK     185
SteamVR Collectibles     46
Chat Text-to-speech      35
Chat Speech-to-text      27
Mods                     10
Mods (require HL2)        9
Name: count, dtype: int64


## 5. Duplicate Check

Verifying that each application appears once and investigating repeated names.

In [20]:
print("Duplicate appids:", applications['appid'].duplicated().sum())
print("Fully duplicated rows:", applications.duplicated().sum())

for label, df in [('application_developers', application_developers),
                  ('application_publishers', application_publishers),
                  ('application_platforms', application_platforms)]:
    print(f"{label} duplicate rows:", df.duplicated().sum())

Duplicate appids: 0
Fully duplicated rows: 0
application_developers duplicate rows: 0
application_publishers duplicate rows: 0
application_platforms duplicate rows: 0


In [21]:
dup_names = applications[applications.duplicated(subset='name', keep=False)]
print(f"Rows sharing a name with another row: {len(dup_names):,}")
dup_names.sort_values('name')[['appid', 'name', 'type', 'release_date', 'mat_initial_price']].head(20)

Rows sharing a name with another row: 3,509


,appid,name,type,release_date,mat_initial_price
174384,2859470,.,game,None,None
83161,1467600,.,game,2020-12-02,99.0
146938,2418320,9 Lives,game,None,None
206644,3378200,9 Lives,game,None,None
184581,3022780,A Better Tomorrow,game,2024-07-31,None
229467,3763930,A Better Tomorrow,game,None,None
58520,1103380,A Long Night For Crazy King Demo,demo,None,None
58605,1104540,A Long Night For Crazy King Demo,demo,2019-06-18,None
41330,842420,A Walk in the Woods,game,2018-05-01,None
22725,557900,A Walk in the Woods,game,2020-04-21,None


In [22]:
   # Strict duplicate check: identical name, type and non-null release_date
strict_cols = ['name', 'type', 'release_date']
suspect = applications[applications['release_date'].notna()
                       & applications.duplicated(subset=strict_cols, keep=False)]

In [23]:
print(f"Suspect rows in total: {len(suspect):,}")
print(suspect['type'].value_counts(dropna=False))

suspect_games = suspect[suspect['type'] == 'game']
print(f"\nSuspect rows with type = 'game': {len(suspect_games):,}")
suspect_games.sort_values('name')[['appid', 'name', 'release_date', 'mat_initial_price']].head(20)

Suspect rows in total: 54
type
dlc      33
None     17
game      2
music     2
Name: count, dtype: int64

Suspect rows with type = 'game': 2


,appid,name,release_date,mat_initial_price
131304,2186000,Nautical Dispute,2023-08-10,None
144073,2377910,Nautical Dispute,2023-08-10,None


**Findings:** `appid` is unique across all 239,664 rows, no row is fully duplicated, and the
junction tables contain no duplicate pairs. About 1.5% of rows (3,509) share a name with
another row, but these are mostly separate Steam listings (different games with the same
title, or a game and a music listing), not duplicate records.

A stricter check on rows sharing name, type and a non-null release date found 54 rows.
52 of them are DLC, music, or untyped listings, which are excluded later when the dataset
is filtered to `type = 'game'`. Only one pair of games remains (*Nautical Dispute*, same
release date, no price). No name-based deduplication is applied; `appid` is treated as
the entity key.

## 6. Categorical Variable Check

Reviewing the distinct values of key categorical columns before filtering and type conversion.

In [24]:
for col in ['type', 'is_free', 'required_age']:
    print(applications[col].value_counts(dropna=False), '\n')

applications['mat_currency'].value_counts(dropna=False).head(8)

type
game     150279
dlc       53792
demo      23521
music      8713
None       2667
video       692
Name: count, dtype: int64 

is_free
False    191390
True      48274
Name: count, dtype: int64 

required_age
0                                                                             235672
17                                                                              2610
18                                                                               581
13                                                                               509
16                                                                                85
12                                                                                60
10                                                                                55
14                                                                                30
7                                                                                 11
15                       

mat_currency
USD     142570
None     94741
EUR        621
AUD        323
PLN        272
CAD        258
PHP        202
VND        125
Name: count, dtype: int64

In [25]:
flags = ['mat_supports_windows', 'mat_supports_mac', 'mat_supports_linux']
print(applications[flags].apply(lambda s: s.value_counts()).T)

plat = application_platforms.merge(platforms, left_on='platform_id', right_on='id')
print(plat['name'].value_counts())

                        True  False
mat_supports_windows  239658      6
mat_supports_mac      239392    272
mat_supports_linux    239364    300
name
windows    239584
mac         55390
linux       39697
Name: count, dtype: int64


In [26]:
platform_ids = {'windows': '1', 'mac': '2', 'linux': '3'}

for name, pid in platform_ids.items():
    in_junction = applications['appid'].isin(
        application_platforms.loc[application_platforms['platform_id'] == pid, 'appid']
    )
    flag = applications[f'mat_supports_{name}'].astype(str).eq('True')
    print(name)
    print(pd.crosstab(flag.rename('flag says yes'), in_junction.rename('junction says yes')), '\n')

windows
junction says yes  False   True 
flag says yes                   
False                  0       6
True                  80  239578 

mac
junction says yes   False  True 
flag says yes                   
False                 239     33
True               184035  55357 

linux
junction says yes   False  True 
flag says yes                   
False                 262     38
True               199705  39659 



- Platforms: `application_platforms` is the source of truth. The `mat_supports_*` flags are dropped:
  they agree with the junction table for Windows (99.96%), but are `True` for ~99.9% of apps on
  Mac and Linux, contradicting the junction table's 23% and 17% (about 184,000 and 200,000
  disagreements respectively).
- Example: *ELDEN RING* (Windows-only on Steam) has `mat_supports_mac` and `mat_supports_linux`
  set to True, while `application_platforms` lists only Windows.

## 7. Numerical Variable Check

Checking that numeric columns convert cleanly and fall in plausible ranges.
Conversion here is for inspection only; the actual type conversion happens in the cleaning section.
- 31% of non-free games (40,974 of 131,285) have no price. Missing prices are strongly tied to a
  missing release date: among non-free games, 90.4% with a release date have a price versus
  0.1% without one. Two groups stand out: 31,380 games with no date and no price (nearly all have a
  description, so they are not blank stubs) and 9,594 games with a date but no price (only 22%
  have a description). Missing prices stay null (never 0), and price analysis covers only games
  with a USD price. Missing dates likewise stay null.
- The extreme raw price maximum is an IDR listing. The USD maximum ($1,900) and other top prices
  look like professional software; they are kept as genuine outliers.

In [27]:
num_cols = ['metacritic_score', 'recommendations_total', 'mat_initial_price',
            'mat_final_price', 'mat_discount_percent', 'mat_achievement_count']

numeric_view = applications[num_cols].apply(pd.to_numeric, errors='coerce')

# Non-null text values that failed to convert would indicate dirty entries
print((applications[num_cols].notna() & numeric_view.isna()).sum())

numeric_view.describe(percentiles=[.01, .5, .99]).T

metacritic_score         0
recommendations_total    0
mat_initial_price        0
mat_final_price          0
mat_discount_percent     0
mat_achievement_count    0
dtype: int64


,count,mean,std,min,1%,50%,99%,max
metacritic_score,5299.0,74.052651,10.299116,6.0,41.98,76.0,91.02,97.0
recommendations_total,22560.0,4648.428989,46422.733998,101.0,103.00,369.0,77081.48,4661011.0
mat_initial_price,144923.0,22651.393161,857183.148742,50.0,99.00,499.0,6999.00,89900000.0
mat_final_price,144923.0,20648.692430,779602.157919,44.0,49.00,499.0,5999.00,89100000.0
mat_discount_percent,144923.0,5.361916,17.769805,0.0,0.00,0.0,85.00,95.0
mat_achievement_count,61401.0,34.411215,197.958850,0.0,0.00,17.0,137.00,9821.0


In [28]:
usd = applications['mat_currency'] == 'USD'
p = numeric_view[usd]

print("final > initial:", (p['mat_final_price'] > p['mat_initial_price']).sum())
print("discount outside 0-100:", ((p['mat_discount_percent'] < 0) | (p['mat_discount_percent'] > 100)).sum())
print("metacritic outside 0-100:", ((numeric_view['metacritic_score'] < 0) | (numeric_view['metacritic_score'] > 100)).sum())
print(p['mat_initial_price'].value_counts().head(8))

final > initial: 0
discount outside 0-100: 0
metacritic outside 0-100: 0
mat_initial_price
499.0     18996
299.0     15963
99.0      15892
999.0     14960
199.0     14157
399.0      8525
1999.0     6951
1499.0     6835
Name: count, dtype: int64


In [29]:
priced = applications.assign(price=numeric_view['mat_initial_price'])
print(priced.nlargest(5, 'price')[['name', 'type', 'mat_currency', 'price']])
print(priced.loc[usd, 'price'].describe(percentiles=[.5, .99, .999]))

                                          name  type mat_currency       price
6899    Suicide Squad: Kill the Justice League  game          IDR  89900000.0
119326                 Call of Duty®: Vanguard  game          IDR  89100000.0
50985                          Hogwarts Legacy  game          IDR  79900000.0
105762             DRAGON BALL: Sparking! ZERO  game          IDR  69900000.0
121222                             LIVE A LIVE  game          IDR  64900000.0
count    142570.000000
mean        917.559753
std        1580.916138
min          50.000000
50%         499.000000
99%        4999.000000
99.9%     19999.000000
max      190000.000000
Name: price, dtype: float64


In [30]:
paid = (applications[applications['is_free'].astype(str) == 'False']
        .assign(has_price=lambda d: d['mat_initial_price'].notna()))

print(pd.crosstab(paid['type'].fillna('NULL'), paid['has_price']))
print(paid.groupby(paid['release_date'].notna().rename('has_release_date'))['has_price'].mean().round(3))

has_price  False  True 
type                   
NULL        2143      9
dlc         2353  46941
game       40974  90311
music        330   7662
video        667      0
has_release_date
False    0.012
True     0.910
Name: has_price, dtype: float64


In [31]:
games = applications[applications['type'] == 'game'].copy()
paid_games = games[games['is_free'].astype(str) == 'False'].copy()

paid_games['has_date'] = paid_games['release_date'].notna()
paid_games['has_price'] = paid_games['mat_initial_price'].notna()
paid_games['has_description'] = paid_games['short_description'].notna()

print(pd.crosstab(paid_games['has_date'], paid_games['has_price']))
print(paid_games.groupby(['has_date', 'has_price'])['has_description'].mean().round(3))

print(priced.loc[usd].nlargest(5, 'price')[['name', 'type', 'price']])

has_price  False  True 
has_date               
False      31380     39
True        9594  90272
has_date  has_price
False     False        0.996
          True         1.000
True      False        0.220
          True         1.000
Name: has_description, dtype: float64
                                                   name  type     price
164111                                    MolCollabo v2  game  190000.0
152229                                The Leverage Game  game   99998.0
152473               The Leverage Game Business Edition  game   99998.0
65379                 Ascent Free-Roaming VR Experience  game   99900.0
33511   Strata Spaces VR – Professional Edition Upgrade   dlc   99500.0


## 8. Date and Time Columns

Parsing date columns and checking that ranges and formats are plausible.

In [32]:
rd = applications['release_date']

print("Non-null:", rd.notna().sum())
print("Not in YYYY-MM-DD format:", (rd.notna() & ~rd.astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$')).sum())

release = pd.to_datetime(rd, errors='coerce')
print("Failed to parse:", (rd.notna() & release.isna()).sum())
print("Range:", release.min(), "->", release.max())

Non-null: 202809
Not in YYYY-MM-DD format: 0
Failed to parse: 2
Range: 1969-12-31 00:00:00 -> 2099-12-31 00:00:00


In [33]:
years = release.dt.year
print("Before 2003:", (years < 2003).sum())
print("After 2025-09-29:", (release > '2025-09-29').sum())
print(years.value_counts().sort_index().tail(12))

for col in ['created_at', 'updated_at']:
    ts = pd.to_datetime(applications[col], errors='coerce', utc=True)
    print(col, ts.min(), ts.max(), "unparseable:", ts.isna().sum())

Before 2003: 15
After 2025-09-29: 904
release_date
2030.0    5
2031.0    3
2032.0    1
2034.0    1
2039.0    1
2040.0    2
2052.0    1
2069.0    4
2077.0    3
2080.0    2
2096.0    1
2099.0    1
Name: count, dtype: int64
created_at 2025-09-07 12:36:43.243384+00:00 2025-09-07 16:27:12.218587+00:00 unparseable: 0
updated_at 2025-09-29 02:01:37.107239+00:00 2025-09-29 06:05:08.643993+00:00 unparseable: 0


In [34]:
scrape = pd.Timestamp('2025-09-29')   # matches the updated_at timestamps

# 1. The two unparseable values
bad = applications[rd.notna() & release.isna()]
print(bad[['appid', 'name', 'type', 'release_date']])

# 2. Future dates by year
print(release[release > scrape].dt.year.value_counts().sort_index().to_string())

# 3. What kind of listings have future dates?
future = applications[release > scrape]
print(future['type'].value_counts(dropna=False))
print("Share with a description:", future['short_description'].notna().mean().round(3))
print(future.sort_values('release_date', ascending=False)[['appid', 'name', 'type', 'release_date']].head(10))

# 4. The earliest dates (1969/1970 and other pre-2003 values)
early = applications[release < '2003-01-01']
print(early[['appid', 'name', 'type', 'release_date']].sort_values('release_date').head(15))

          appid                      name  type release_date
100059  1706960  The Mysterious Cat Tower  game   6969-06-09
132379  2201700           Beyond The Lens  game   9998-12-31
release_date
2025    702
2026    154
2027     10
2028      9
2029      4
2030      5
2031      3
2032      1
2034      1
2039      1
2040      2
2052      1
2069      4
2077      3
2080      2
2096      1
2099      1
type
game     836
demo      40
dlc       23
music      5
Name: count, dtype: int64
Share with a description: 0.979
          appid                                               name  type  \
221878  3633220                      Aletheia: Prophecy of Perseus  game   
126275  2112370                                 World of Yggdrasil  game   
172904  2835840  "The Hardest Game in the Universe 2: Bridging ...   dlc   
140142  2319410                                          WhyTaoism  game   
146965  2418840                                         Schoolmare  game   
29597    655750              

In [35]:
# 1. Release counts by year, restricted to valid observed dates
d = (applications.assign(release=release)
     .loc[lambda x: (x['release'] >= '1990-01-01') & (x['release'] <= scrape)]
     .assign(year=lambda x: x['release'].dt.year))
by_year = d.groupby('year').agg(all_types=('appid', 'size'),
                                games=('type', lambda s: (s == 'game').sum()))
print(by_year.tail(12))

# 2. Do paid games with no date/price look like recent listings? (appid rises with time roughly)
games_all = applications[applications['type'] == 'game']
print("Games with no release_date:", games_all['release_date'].isna().sum(), "of", len(games_all))

g = games_all[games_all['is_free'].astype(str) == 'False'].assign(
    appid_n=lambda x: pd.to_numeric(x['appid']),
    has_date=lambda x: x['release_date'].notna(),
    has_price=lambda x: pd.to_numeric(x['mat_initial_price'], errors='coerce').notna(),
)
print(g.groupby(['has_date', 'has_price'])['appid_n'].describe()[['count', '25%', '50%', '75%']])

      all_types  games
year                  
2014       2687   1515
2015       4568   2502
2016       8660   4141
2017      11275   5926
2018      13102   7438
2019      12742   7254
2020      15728   8791
2021      20083  11066
2022      21103  12311
2023      24567  14664
2024      33780  20203
2025      29744  17236
Games with no release_date: 34483 of 150279
                      count        25%        50%        75%
has_date has_price                                          
False    False      31380.0  2133770.0  2903160.0  3487275.0
         True          39.0   219650.0   534380.0  1067420.0
True     False       9594.0  2008925.0  2792160.0  3455425.0
         True       90272.0   941615.0  1689115.0  2585042.5


In [36]:
nop = games_all[(games_all['is_free'].astype(str) == 'False')
                & pd.to_numeric(games_all['mat_initial_price'], errors='coerce').isna()]
print("Unpriced paid games:", len(nop), "| future-dated:", (nop['release_date'] > '2025-09-29').sum())

Unpriced paid games: 40974 | future-dated: 748


- Release counts rise from 1,515 games (2014) to 20,203 (2024); 2025 (17,236) covers only 1 January to 29 September.
- Games with no release date (34,483) and games with no price skew toward recent appids. Only 748 of 40,974 unpriced paid games (1.8%) are future-dated, so "unreleased" does not explain the missing prices; the cause is undetermined. Price-based analysis therefore describes a subset biased toward older listings; documented as a limitation.

## 9. Relational Integrity

In [37]:
app_ids = set(applications['appid'])

junction_checks = {
    'application_genres':           (application_genres,           None,           None),
    'application_categories_clean': (application_categories_clean, None,           None),
    'application_developers':       (application_developers,       'developer_id', developers),
    'application_publishers':       (application_publishers,       'publisher_id', publishers),
    'application_platforms':        (application_platforms,        'platform_id',  platforms),
}

for name, (df, fk_col, lookup) in junction_checks.items():
    line = (f"{name}: {len(df):,} rows | null appid: {df['appid'].isna().sum()} "
            f"| appid not in applications: {(~df['appid'].isin(app_ids)).sum():,}")
    if fk_col:
        line += (f" | null {fk_col}: {df[fk_col].isna().sum()} "
                 f"| {fk_col} not in lookup: {(~df[fk_col].isin(set(lookup['id']))).sum():,}")
    print(line)

application_genres: 587,514 rows | null appid: 0 | appid not in applications: 0
application_categories_clean: 1,073,302 rows | null appid: 0 | appid not in applications: 0
application_developers: 250,817 rows | null appid: 0 | appid not in applications: 0 | null developer_id: 0 | developer_id not in lookup: 0
application_publishers: 223,048 rows | null appid: 0 | appid not in applications: 0 | null publisher_id: 0 | publisher_id not in lookup: 0
application_platforms: 334,671 rows | null appid: 0 | appid not in applications: 0 | null platform_id: 0 | platform_id not in lookup: 0


In [38]:
game_apps = applications[applications['type'] == 'game']
for name, (df, _, _) in junction_checks.items():
    missing = ~game_apps['appid'].isin(set(df['appid']))
    print(f"{name}: games with no rows = {missing.sum():,} ({missing.mean():.1%} of {len(game_apps):,})")

application_genres: games with no rows = 7,723 (5.1% of 150,279)
application_categories_clean: games with no rows = 8,331 (5.5% of 150,279)
application_developers: games with no rows = 7,742 (5.2% of 150,279)
application_publishers: games with no rows = 8,259 (5.5% of 150,279)
application_platforms: games with no rows = 0 (0.0% of 150,279)


In [39]:
win_flag = applications['mat_supports_windows'].astype(str).str.lower() == 'true'
win_ids = set(application_platforms.loc[application_platforms['platform_id'] == '1', 'appid'])
any_platform = set(application_platforms['appid'])

flag_no_row = applications[win_flag & ~applications['appid'].isin(win_ids)]
row_no_flag = applications[~win_flag & applications['appid'].isin(win_ids)]
print("Windows flag True, no Windows junction row:", len(flag_no_row))
print("  ...of which have no platform rows at all:", (~flag_no_row['appid'].isin(any_platform)).sum())
print(flag_no_row['type'].value_counts(dropna=False))
print("Windows junction row, flag not True:", len(row_no_flag))
print(flag_no_row[['appid', 'name', 'type']].head(10))

Windows flag True, no Windows junction row: 80
  ...of which have no platform rows at all: 0
type
game    61
demo    10
dlc      9
Name: count, dtype: int64
Windows junction row, flag not True: 6
        appid                                               name  type
2856   214630              Call of Duty: Black Ops - Mac Edition  game
2857   214643  Call of Duty®: Black Ops Annihilation & Escala...   dlc
2858   214649  Call of Duty: Black Ops - Rezurrection Mac Edi...   dlc
9942   364380                                CINEVEO - VR Cinema  game
12200  398280                Tidal Affair: Before The Storm Demo  demo
14941  443320              Family Tree Heritage Platinum 9 - Mac  game
16086  461530                                   CV Maker for Mac  game
16088  461550                Typing Instructor Platinum 21 - Mac  game
17594  492520                              Resume Maker® for Mac  game
23471  569050                         Paul Pixel - The Awakening  game


In [40]:
for name, lookup, junction, fk in [('developers', developers, application_developers, 'developer_id'),
                                   ('publishers', publishers, application_publishers, 'publisher_id')]:
    norm = lookup['name'].str.strip().str.lower()
    print(f"{name}: {len(lookup):,} rows | unused ids: {(~lookup['id'].isin(set(junction[fk]))).sum():,} "
          f"| blank names: {(norm.isna() | (norm == '')).sum()} "
          f"| rows sharing a normalized name: {norm.duplicated(keep=False).sum():,}")
    print(lookup[norm.duplicated(keep=False)].assign(n=norm).sort_values('n').head(8)[['id', 'name']])

developers: 101,226 rows | unused ids: 4 | blank names: 1 | rows sharing a normalized name: 2,955
          id               name
38076  38077              0Cube
67933  67934              0cube
15362  15363  100 percent games
36583  36584  100 Percent Games
29953  29954            126EAST
19582  19583            126east
85810  85811    150 Beanies LTD
44084  44085    150 beanies LTD
publishers: 85,699 rows | unused ids: 4 | blank names: 1 | rows sharing a normalized name: 4,145
          id             name
55088  55089      072 Project
35311  35312      072 Project
39760  39761      072 PROJECT
32200  32201            0Cube
57528  57529            0cube
25342  25343          126EAST
16548  16549          126east
37326  37327  150 beanies LTD


In [41]:
# 1. Do the games missing relational metadata overlap, and do they relate to missing date/price?
g = game_apps.assign(
    no_genre=~game_apps['appid'].isin(set(application_genres['appid'])),
    no_cat=~game_apps['appid'].isin(set(application_categories_clean['appid'])),
    no_dev=~game_apps['appid'].isin(set(application_developers['appid'])),
    no_pub=~game_apps['appid'].isin(set(application_publishers['appid'])),
    no_date=game_apps['release_date'].isna(),
    no_price=pd.to_numeric(game_apps['mat_initial_price'], errors='coerce').isna()
             & (game_apps['is_free'].astype(str) == 'False'),
)
meta = ['no_genre', 'no_cat', 'no_dev', 'no_pub']
print("Missing at least one of the four:", g[meta].any(axis=1).sum(),
      "| missing all four:", g[meta].all(axis=1).sum())
print(g.groupby(g[meta].all(axis=1).rename('missing_all_four'))[['no_date', 'no_price']].mean().round(3))

# 2. How many IDs would merge on strip + lowercase?
for name, lookup in [('developers', developers), ('publishers', publishers)]:
    norm = lookup['name'].str.strip().str.lower()
    n = norm.nunique(dropna=False)
    print(f"{name}: {len(lookup):,} ids -> {n:,} normalized names ({len(lookup) - n:,} ids would merge)")

# 3. Blank names, and the identical-looking '072 Project' pair
for name, lookup, junction, fk in [('developers', developers, application_developers, 'developer_id'),
                                   ('publishers', publishers, application_publishers, 'publisher_id')]:
    blank = lookup[lookup['name'].isna() | (lookup['name'].str.strip() == '')]
    print(name, "blank-name ids:", blank['id'].tolist(),
          "| junction rows using them:", junction[fk].isin(blank['id']).sum())
print([repr(n) for n in publishers.loc[publishers['id'].isin(['55089', '35312']), 'name']])

Missing at least one of the four: 9521 | missing all four: 7266
                  no_date  no_price
missing_all_four                   
False               0.241     0.236
True                0.011     1.000
developers: 101,226 ids -> 99,732 normalized names (1,494 ids would merge)
publishers: 85,699 ids -> 83,557 normalized names (2,142 ids would merge)
developers blank-name ids: ['22936'] | junction rows using them: 1
publishers blank-name ids: ['19454'] | junction rows using them: 11
["'072 Project'", "'072 Project'"]


**Findings:** No null or orphaned keys in any junction table. About 5% of games lack rows in each of genres, categories, developers and publishers; 7,266 lack all four, and essentially all of them are paid games with no price (98.9% have a release date). Platform rows exist for every game. 80 apps flagged as Windows-supported have no Windows platform row (all have rows for other platforms), supporting `application_platforms` as the source of truth. Developer and publisher lookups contain 1,494 and 2,142 IDs whose names duplicate another ID's name up to case and whitespace (some exactly), plus one blank-named ID each (used by 1 and 11 junction rows).

**Decisions carried into cleaning:**
- Keep games that lack relational rows; add a boolean `has_relational_metadata` (false when a game has none of genre, category, developer or publisher rows) so analysis can filter explicitly.
- Merge developer and publisher IDs that share a normalized (trimmed, lowercased) name; canonical spelling is the variant linked to the most games (ties: lowest ID). Remove duplicate (appid, id) pairs created by the merge.
- Drop junction rows that point to blank-named developers or publishers; keep the games.
- Restrict lookup tables to IDs referenced by retained games.

### 10. Cleaning

### 10.1 Scope to games
#### Restricting to games

The analysis is about games. DLC, demos, soundtracks and videos are different entities with different pricing and review behaviour, and 2,667 rows have no type at all, so their nature is unknown. Only rows with `type = 'game'` are kept (239,664 to 150,279 rows). Excluded rows are not deleted from the source database.

In [42]:
games_clean = applications[applications['type'] == 'game'].copy()
print(f"applications: {len(applications):,} -> games_clean: {len(games_clean):,}")
print(applications['type'].value_counts(dropna=False))

applications: 239,664 -> games_clean: 150,279
type
game     150279
dlc       53792
demo      23521
music      8713
None       2667
video       692
Name: count, dtype: int64


### 10.2 Column types

Every column arrived as text from the CSV import. `is_free` becomes a boolean (no nulls in the source). The count and price columns are converted to numeric. Section 7 showed every value converts, so the cell asserts that no new nulls appear rather than silently coercing. Integer casting is deferred to the export step, so that missing values stay explicit.

In [43]:
games_clean['is_free'] = (games_clean['is_free'].astype(str).str.lower()
                          .map({'true': True, 'false': False}))
assert games_clean['is_free'].notna().all()

numeric_cols = ['metacritic_score', 'recommendations_total', 'mat_initial_price',
                'mat_final_price', 'mat_discount_percent', 'mat_achievement_count']
before = games_clean[numeric_cols].isna().sum()
converted = games_clean[numeric_cols].apply(pd.to_numeric, errors='coerce')
assert ((converted.isna().sum() - before) == 0).all(), "conversion introduced nulls"
games_clean[numeric_cols] = converted

print(games_clean['is_free'].value_counts())
print(games_clean[numeric_cols].dtypes)

# Free games and prices: null, zero, or something else?
free_prices = games_clean.loc[games_clean['is_free'], 'mat_initial_price']
print("Free games:", len(free_prices), "| with a price:", free_prices.notna().sum(),
      "| with a price > 0:", (free_prices > 0).sum())

is_free
False    131285
True      18994
Name: count, dtype: int64
metacritic_score         float64
recommendations_total    float64
mat_initial_price        float64
mat_final_price          float64
mat_discount_percent     float64
mat_achievement_count    float64
dtype: object
Free games: 18994 | with a price: 0 | with a price > 0: 0


### 10.3 Prices

Prices are stored in the smallest currency unit (cents for USD) and mix 33 currencies, so raw values are not comparable. USD prices (98% of priced rows) are converted to dollars in `price_usd` and `price_final_usd`; prices in other currencies are set to null rather than converted, because the dataset contains no exchange rates and an external rate would add an unverifiable assumption. `mat_currency` and `mat_discount_percent` are kept for all rows (a discount percentage is currency-independent). Missing prices remain null, never 0. Free games have null prices in the source, so `is_free` is the only marker of free games.

In [44]:
is_usd = games_clean['mat_currency'] == 'USD'

games_clean['price_usd'] = (games_clean['mat_initial_price'] / 100).where(is_usd)
games_clean['price_final_usd'] = (games_clean['mat_final_price'] / 100).where(is_usd)

# 1. Are currency and price missing together, and initial/final consistent?
print(pd.crosstab(games_clean['mat_currency'].isna(), games_clean['mat_initial_price'].isna(),
                  rownames=['currency null'], colnames=['price null']))
print("initial/final null mismatch:",
      (games_clean['mat_initial_price'].isna() != games_clean['mat_final_price'].isna()).sum())

# 2. Funnel
paid = games_clean[~games_clean['is_free']]
print("Games:", len(games_clean))
print("  paid:", len(paid))
print("  paid with any price:", paid['mat_initial_price'].notna().sum())
print("  paid with USD price:", paid['price_usd'].notna().sum())
print("  paid with price == 0:", (paid['price_usd'] == 0).sum())

# 3. Sanity
both = games_clean['price_usd'].notna() & games_clean['price_final_usd'].notna()
assert (games_clean.loc[both, 'price_final_usd'] <= games_clean.loc[both, 'price_usd']).all()
print(games_clean['price_usd'].describe(percentiles=[.5, .99]))

price null     False  True 
currency null              
False          90311      0
True               0  59968
initial/final null mismatch: 0
Games: 150279
  paid: 131285
  paid with any price: 90311
  paid with USD price: 88943
  paid with price == 0: 0
count    88943.000000
mean         9.489877
std         17.088101
min          0.500000
50%          5.990000
99%         54.990000
max       1900.000000
Name: price_usd, dtype: float64


### 10.4 Results
#### Required age

`required_age` is 0 for 98.3% of rows but contains scraped page code (a `javascript:` snippet), `17+` and an impossible value (120). Values with a trailing `+` are converted to the number; anything not numeric or outside 0-21 becomes null (unknown), not 0, because a missing age rating is not the same as "no age restriction". The cell prints every value that was nulled so the rule can be audited.

In [45]:
raw_age = applications.loc[games_clean.index, 'required_age']
print("Raw distinct values:")
print(raw_age.astype(str).str[:30].value_counts(dropna=False))

age = pd.to_numeric(raw_age.astype(str).str.strip().str.rstrip('+'), errors='coerce')
age = age.where(age.between(0, 21))
games_clean['required_age'] = age.astype('Int64')

print("\nNulled by the rule (raw values):")
print(raw_age[age.isna() & raw_age.notna()].astype(str).str[:40].value_counts())
print("\nCleaned distribution:")
print(games_clean['required_age'].value_counts(dropna=False).sort_index())

Raw distinct values:
required_age
0                                 148947
17                                   811
18                                   204
13                                   180
16                                    41
10                                    31
12                                    25
15                                     7
3                                      6
7                                      6
14                                     6
6                                      4
1                                      2
21                                     2
17+                                    2
11                                     1
120                                    1
20                                     1
javascript:ToggleCheckbox('che         1
8                                      1
Name: count, dtype: int64

Nulled by the rule (raw values):
required_age
120                                         1
javascript:ToggleCheckbox('

In [46]:
name = games_clean['name']
print("Null names:", name.isna().sum())
no_word = name.notna() & ~name.str.contains(r'\w')
print("Names with no letter or digit:", no_word.sum())
print(games_clean.loc[no_word, ['appid', 'name', 'short_description']]
      .assign(short_description=lambda x: x['short_description'].str[:50]).head(20))

Null names: 0
Names with no letter or digit: 7
          appid name                                  short_description
50453    981470    ※                                               None
53912   1034230  ***  *** is a small game. It is easy to play,hope y...
83161   1467600    .                                               None
117866  1963980  ...  In a universe where there are 4 realms (Bright...
174384  2859470    .  A co-op horror about searching the depths of t...
179870  2948240    ∞  This is a role-playing sci-fi game.Use the A a...
192349  3147470    ∆  ∆ is an atmospheric exploration experience tha...


#### Release dates

Following Section 8: dates before 1990-01-01 or after the scrape date (2025-09-29) are set to null; the listing is retained. `is_unreleased` marks games whose raw date lies after the scrape date. Comparisons use ISO strings, which sort chronologically and also handle values beyond pandas' timestamp range (6969-06-09, 9998-12-31). `release_year` is derived from valid dates only, and 2025 is a partial year.

In [47]:
SCRAPE_DATE = '2025-09-29'
raw_rd = applications.loc[games_clean.index, 'release_date'].fillna('')

valid = (raw_rd >= '1990-01-01') & (raw_rd <= SCRAPE_DATE)
games_clean['is_unreleased'] = raw_rd > SCRAPE_DATE
games_clean['release_date'] = raw_rd.where(valid)
games_clean['release_year'] = pd.to_numeric(games_clean['release_date'].str[:4]).astype('Int64')

print("Raw dates present:", (raw_rd != '').sum())
print("is_unreleased:", games_clean['is_unreleased'].sum())
print("Nulled as out of range:", ((raw_rd != '') & ~valid).sum())
print("Valid release_date:", games_clean['release_date'].notna().sum(),
      "| null:", games_clean['release_date'].isna().sum())
print(games_clean['release_year'].value_counts().sort_index().tail(3))

Raw dates present: 115796
is_unreleased: 838
Nulled as out of range: 841
Valid release_date: 114955 | null: 35324
release_year
2023    14664
2024    20203
2025    17236
Name: count, dtype: Int64


### Names

Seven game names contain no letter or digit (for example `.`, `***`, `∞`). Each has a valid appid, and five have a real description, so they are treated as legitimate, if unusual, titles. Names are left unchanged; `appid` remains the entity key.

## 10.5 Dropping and renaming columns

Dropped: `type` (constant after filtering), `mat_supports_*` (contradicted by the platform table, Section 6), `header_image` and `background` (image URLs), `created_at` and `updated_at` (scrape metadata, Section 8), and the raw `mat_initial_price` / `mat_final_price` (superseded by the USD dollar columns). The `mat_` prefix is removed from the remaining columns for readability.

In [48]:
assert set(games_clean['type']) == {'game'}
drop_cols = ['type', 'mat_supports_windows', 'mat_supports_mac', 'mat_supports_linux',
             'header_image', 'background', 'created_at', 'updated_at',
             'mat_initial_price', 'mat_final_price']
games_clean = games_clean.drop(columns=drop_cols)
games_clean = games_clean.rename(columns=lambda c: c[4:] if c.startswith('mat_') else c)

print(games_clean.shape)
print(games_clean.columns.tolist())

(150279, 24)
['appid', 'name', 'is_free', 'release_date', 'required_age', 'short_description', 'supported_languages', 'metacritic_score', 'recommendations_total', 'discount_percent', 'currency', 'achievement_count', 'pc_os_min', 'pc_processor_min', 'pc_memory_min', 'pc_graphics_min', 'pc_os_rec', 'pc_processor_rec', 'pc_memory_rec', 'pc_graphics_rec', 'price_usd', 'price_final_usd', 'is_unreleased', 'release_year']


### 10.6a: Restrict junctions and merge developers and publishers
#### Developers and publishers

Junction rows are restricted to retained games. Links to the blank-named developer and publisher IDs are dropped (the games are kept). IDs sharing a trimmed, lowercased name are merged into one ID; the canonical spelling is that of the variant linked to the most games (ties: lowest ID). Duplicate (appid, id) pairs created by the merge are removed. Only case and whitespace variants are merged; other name variants and distinct companies that share a name cannot be told apart from this data and are left as they are (README limitation).

In [49]:
game_ids = set(games_clean['appid'])

def merge_lookup(lookup, junction, fk):
    j = junction[junction['appid'].isin(game_ids)]
    lk = lookup.assign(id_n=lookup['id'].astype(int),
                       norm=lookup['name'].str.strip().str.lower())
    lk = lk[lk['norm'].notna() & (lk['norm'] != '')]              # blank names out
    lk['n_games'] = lk['id'].map(j.groupby(fk)['appid'].nunique()).fillna(0)
    canon = (lk.sort_values(['n_games', 'id_n'], ascending=[False, True])
               .drop_duplicates('norm')[['norm', 'id', 'name']]
               .rename(columns={'id': 'canon_id', 'name': 'canon_name'}))
    id_map = dict(zip(lk['id'], lk.merge(canon, on='norm')['canon_id']))

    kept = j[j[fk].isin(id_map)]
    merged = kept.assign(**{fk: kept[fk].map(id_map)})
    clean = merged.drop_duplicates(['appid', fk]).reset_index(drop=True)
    lookup_clean = (canon[canon['canon_id'].isin(set(clean[fk]))]
                    .rename(columns={'canon_id': 'id', 'canon_name': 'name'})[['id', 'name']])
    print(f"{fk}: game rows {len(j):,} | blank-name links dropped {len(j) - len(kept):,} "
          f"| duplicate pairs removed {len(merged) - len(clean):,} | final rows {len(clean):,}")
    print(f"  lookup ids: {len(lookup):,} -> {len(lookup_clean):,}")
    return clean, lookup_clean

dev_links, dev_lookup = merge_lookup(developers, application_developers, 'developer_id')
pub_links, pub_lookup = merge_lookup(publishers, application_publishers, 'publisher_id')

# Assertions
for links, lk, fk in [(dev_links, dev_lookup, 'developer_id'), (pub_links, pub_lookup, 'publisher_id')]:
    assert not links.duplicated(['appid', fk]).any()
    assert links[fk].isin(set(lk['id'])).all() and links['appid'].isin(game_ids).all()
    assert lk['name'].str.strip().str.lower().is_unique

# Spot check the merge on two known variants
for n in ['0cube', '126east']:
    print(n, dev_lookup[dev_lookup['name'].str.strip().str.lower() == n].to_dict('records'))

developer_id: game rows 155,271 | blank-name links dropped 1 | duplicate pairs removed 4 | final rows 155,266
  lookup ids: 101,226 -> 94,983
publisher_id: game rows 147,437 | blank-name links dropped 8 | duplicate pairs removed 2 | final rows 147,427
  lookup ids: 85,699 -> 80,733
0cube [{'id': '38077', 'name': '0Cube'}]
126east [{'id': '19583', 'name': '126east'}]


### 10.6b: Genres, categories, platforms and the metadata flag

Junction rows are restricted to retained games. Genres get a lookup table with surrogate IDs (assigned alphabetically over all 34 canonical names, so IDs do not depend on filtering). Category links reuse the consolidated IDs from Section 6. `has_relational_metadata` is false for games with no genre, category, developer or publisher row at all; games missing only some of these tables are kept as they are, and analyses that need a table use an inner join to it. Every game has at least one platform row.

In [50]:
# Genres: surrogate ids over all canonical names, then restrict to games
genre_lookup = pd.DataFrame({'name': sorted(application_genres['genre_canonical'].unique())})
genre_lookup['id'] = range(1, len(genre_lookup) + 1)
genre_id_map = dict(zip(genre_lookup['name'], genre_lookup['id']))
genre_links = (application_genres[application_genres['appid'].isin(game_ids)]
               .assign(genre_id=lambda d: d['genre_canonical'].map(genre_id_map))
               [['appid', 'genre_id']].reset_index(drop=True))
genre_lookup = genre_lookup[genre_lookup['id'].isin(set(genre_links['genre_id']))][['id', 'name']]

# Categories
cat_links = (application_categories_clean[application_categories_clean['appid'].isin(game_ids)]
             [['appid', 'category_id']].reset_index(drop=True))
cat_lookup = (categories_english[['id', 'name']]
              .loc[lambda d: d['id'].isin(set(cat_links['category_id']))].reset_index(drop=True))

# Platforms
platform_links = application_platforms[application_platforms['appid'].isin(game_ids)].reset_index(drop=True)
platform_lookup = platforms.copy()

# Integrity assertions
for links, lk, fk in [(genre_links, genre_lookup, 'genre_id'),
                      (cat_links, cat_lookup, 'category_id'),
                      (platform_links, platform_lookup, 'platform_id')]:
    assert not links.duplicated(['appid', fk]).any()
    assert links['appid'].isin(game_ids).all() and links[fk].isin(set(lk['id'])).all()
assert platform_links['appid'].nunique() == len(game_ids)

# Metadata flag
has_any = set(pd.concat([genre_links['appid'], cat_links['appid'],
                         dev_links['appid'], pub_links['appid']]))
games_clean['has_relational_metadata'] = games_clean['appid'].isin(has_any)

# Reconciliation
recon = pd.DataFrame([
    ('applications -> games_clean', len(applications), len(games_clean)),
    ('application_genres', len(application_genres), len(genre_links)),
    ('application_categories', len(application_categories_clean), len(cat_links)),
    ('application_developers', len(application_developers), len(dev_links)),
    ('application_publishers', len(application_publishers), len(pub_links)),
    ('application_platforms', len(application_platforms), len(platform_links)),
], columns=['table', 'before', 'after'])
recon['dropped'] = recon['before'] - recon['after']
print(recon.to_string(index=False))
print("\nLookups:", {'genres': len(genre_lookup), 'categories': len(cat_lookup),
                     'developers': len(dev_lookup), 'publishers': len(pub_lookup),
                     'platforms': len(platform_lookup)})
print(platform_lookup)
print(games_clean['has_relational_metadata'].value_counts())

                      table  before  after  dropped
applications -> games_clean  239664 150279    89385
         application_genres  587514 414273   173241
     application_categories 1073302 631241   442061
     application_developers  250817 155266    95551
     application_publishers  223048 147427    75621
      application_platforms  334671 194101   140570

Lookups: {'genres': 33, 'categories': 57, 'developers': 94983, 'publishers': 80733, 'platforms': 3}
  id     name
0  1  windows
1  2      mac
2  3    linux
has_relational_metadata
True     143013
False      7266
Name: count, dtype: int64


## 11. Post-Cleaning Validation

Before export, every guarantee the cleaning steps were meant to establish is tested explicitly: key integrity, relational consistency, value ranges, and the counts recorded in earlier sections. The notebook stops here if any check fails.

### 11.1 Validation checks

In [51]:
# 11.1 Post-cleaning validation checks
results = []

def check(name, condition):
    ok = bool(condition)
    results.append((name, ok))
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

def num(s):
    return pd.to_numeric(s, errors='coerce')

g = games_clean
game_keys = set(g['appid'].astype(str))

# --- Keys ---
check("appid non-null", g['appid'].notna().all())
check("appid unique", g['appid'].is_unique)
check("appid convertible to integer", (num(g['appid']).notna() & (num(g['appid']) % 1 == 0)).all())

# --- Link tables and lookups ---
link_tables = {
    'genres':     (genre_links, genre_lookup),
    'categories': (cat_links, cat_lookup),
    'developers': (dev_links, dev_lookup),
    'publishers': (pub_links, pub_lookup),
    'platforms':  (platform_links, platform_lookup),
}
for label, (links, lookup) in link_tables.items():
    id_col = [c for c in links.columns if c != 'appid'][0]
    lk_id = 'id' if 'id' in lookup.columns else lookup.columns[0]
    lk_name = 'name' if 'name' in lookup.columns else lookup.columns[1]

    check(f"{label}: no duplicate (appid, id) pairs", not links.duplicated(subset=['appid', id_col]).any())
    check(f"{label}: all link appids are kept games", set(links['appid'].astype(str)) <= game_keys)
    check(f"{label}: all link ids exist in lookup", set(links[id_col].astype(str)) <= set(lookup[lk_id].astype(str)))
    check(f"{label}: lookup ids unique", lookup[lk_id].is_unique)
    check(f"{label}: lookup names non-blank", lookup[lk_name].fillna('').astype(str).str.strip().ne('').all())
    check(f"{label}: ids convertible to integer",
          (num(links[id_col]).notna().all()) and (num(lookup[lk_id]).notna().all()))

# --- Column types and ranges ---
check("is_free is boolean", g['is_free'].dtype == bool)
d = num(g['discount_percent']).dropna()
check("discount_percent within 0-100", d.between(0, 100).all())
m = num(g['metacritic_score']).dropna()
check("metacritic_score within 0-100", m.between(0, 100).all())
a = num(g['required_age']).dropna()
check("required_age within 0-21", a.between(0, 21).all())
r = num(g['recommendations_total']).dropna()
check("recommendations_total > 0 where present", (r > 0).all())
ac = num(g['achievement_count']).dropna()
check("achievement_count >= 0", (ac >= 0).all())

# --- Prices ---
pu, pf = num(g['price_usd']), num(g['price_final_usd'])
check("price_usd > 0 where present", (pu.dropna() > 0).all())
both = pu.notna() & pf.notna()
check("price_final_usd <= price_usd", (pf[both] <= pu[both]).all())
priced = pu.notna()
check("priced rows are USD (or currency null)", (g.loc[priced, 'currency'].dropna() == 'USD').all())
check("free games have null price", pu[g['is_free']].isna().all())

# --- Dates ---
rd = pd.to_datetime(g['release_date'], errors='coerce')
check("release_date within 1990-01-01..2025-09-29",
      rd.dropna().between('1990-01-01', SCRAPE_DATE).all())
check("unreleased games have null release_date", g.loc[g['is_unreleased'], 'release_date'].isna().all())
check("release_year matches release_date",
      (g['release_year'].isna() == rd.isna()).all() and
      (num(g['release_year'])[rd.notna()] == rd.dt.year[rd.notna()]).all())

# --- Relational flag and coverage ---
any_link = (set(genre_links['appid'].astype(str)) | set(cat_links['appid'].astype(str)) |
            set(dev_links['appid'].astype(str)) | set(pub_links['appid'].astype(str)))
expected_flag = g['appid'].astype(str).isin(any_link)
check("has_relational_metadata matches link tables", (g['has_relational_metadata'] == expected_flag).all())
check("every game has a platform row", game_keys <= set(platform_links['appid'].astype(str)))

# --- Recorded counts from earlier sections ---
check("150,279 games", len(g) == 150_279)
check("18,994 free games", g['is_free'].sum() == 18_994)
check("131,285 paid games", (~g['is_free']).sum() == 131_285)
check("88,943 USD prices", pu.notna().sum() == 88_943)
check("838 unreleased games", g['is_unreleased'].sum() == 838)
check("114,955 valid release dates", rd.notna().sum() == 114_955)

failed = [n for n, ok in results if not ok]
print(f"\n{len(results) - len(failed)} of {len(results)} checks passed")
assert not failed, f"Validation failed: {failed}"

PASS  appid non-null
PASS  appid unique
PASS  appid convertible to integer
PASS  genres: no duplicate (appid, id) pairs
PASS  genres: all link appids are kept games
PASS  genres: all link ids exist in lookup
PASS  genres: lookup ids unique
PASS  genres: lookup names non-blank
PASS  genres: ids convertible to integer
PASS  categories: no duplicate (appid, id) pairs
PASS  categories: all link appids are kept games
PASS  categories: all link ids exist in lookup
PASS  categories: lookup ids unique
PASS  categories: lookup names non-blank
PASS  categories: ids convertible to integer
PASS  developers: no duplicate (appid, id) pairs
PASS  developers: all link appids are kept games
PASS  developers: all link ids exist in lookup
PASS  developers: lookup ids unique
PASS  developers: lookup names non-blank
PASS  developers: ids convertible to integer
PASS  publishers: no duplicate (appid, id) pairs
PASS  publishers: all link appids are kept games
PASS  publishers: all link ids exist in lookup
PAS

### 11.2 Final profile

In [52]:
# 11.2 Final profile of the cleaned data
print("Null rate per column in games_clean (%):")
print((games_clean.isna().mean() * 100).round(1).sort_values(ascending=False).to_string())

print("\nCoverage: games with at least one row per link table")
for label, (links, _) in link_tables.items():
    n = links['appid'].astype(str).nunique()
    print(f"  {label:<11} {n:>8,}  ({n / len(games_clean):.1%})")

print("\nhas_relational_metadata:")
print(games_clean['has_relational_metadata'].value_counts().to_string())

print("\nLink table sizes:")
for label, (links, lookup) in link_tables.items():
    print(f"  {label:<11} links {len(links):>9,}   lookup {len(lookup):>7,}")

print("\nID dtypes (games_clean.appid and link/lookup id columns):")
print("  games.appid", games_clean['appid'].dtype)
for label, (links, lookup) in link_tables.items():
    id_col = [c for c in links.columns if c != 'appid'][0]
    print(f"  {label:<11} link {links[id_col].dtype}   lookup {lookup[lookup.columns[0]].dtype}")

# Which canonical genre appears only on non-games?
print("\nGenres absent from games:", set(genre_id_to_canonical.values()) - set(genre_lookup['name']))

Null rate per column in games_clean (%):
metacritic_score           97.2
recommendations_total      86.8
achievement_count          60.2
pc_graphics_rec            56.9
pc_processor_rec           55.1
pc_memory_rec              54.7
pc_os_rec                  52.6
price_usd                  40.8
price_final_usd            40.8
discount_percent           39.9
currency                   39.9
release_date               23.5
release_year               23.5
pc_graphics_min            19.1
pc_processor_min           13.0
pc_memory_min              12.7
pc_os_min                   7.3
supported_languages         5.1
short_description           5.1
required_age                0.0
appid                       0.0
is_free                     0.0
name                        0.0
is_unreleased               0.0
has_relational_metadata     0.0

Coverage: games with at least one row per link table
  genres       142,556  (94.9%)
  categories   141,948  (94.5%)
  developers   142,537  (94.8%)
  publish

## 12. Export

The validated tables are written to data/processed/steam_clean.sqlite with explicit integer keys, foreign keys and indexes. The export is verified by reopening the file independently. The data/ folder is gitignored, so the file is regenerated by running this notebook.

### 12.1 Build the export tables

In [53]:
# 12.1 Build export tables with explicit integer types
def as_int(s):
    return pd.to_numeric(s, errors='raise').astype('int64')

def lookup_frame(lookup, id_name, name_name):
    cols = ['id', 'name'] if {'id', 'name'} <= set(lookup.columns) else list(lookup.columns[:2])
    out = lookup[cols].copy()
    out.columns = [id_name, name_name]
    out[id_name] = as_int(out[id_name])
    return out.sort_values(id_name).reset_index(drop=True)

def link_frame(links, id_name):
    src = [c for c in links.columns if c != 'appid'][0]
    out = pd.DataFrame({'appid': as_int(links['appid']), id_name: as_int(links[src])})
    return out.sort_values(['appid', id_name]).reset_index(drop=True)

GAME_COLS = ['appid', 'name', 'is_free', 'release_date', 'release_year', 'required_age',
             'short_description', 'supported_languages', 'metacritic_score',
             'recommendations_total', 'discount_percent', 'currency', 'achievement_count',
             'price_usd', 'price_final_usd', 'is_unreleased', 'has_relational_metadata',
             'pc_os_min', 'pc_processor_min', 'pc_memory_min', 'pc_graphics_min',
             'pc_os_rec', 'pc_processor_rec', 'pc_memory_rec', 'pc_graphics_rec']
assert set(GAME_COLS) == set(games_clean.columns), set(games_clean.columns) ^ set(GAME_COLS)

games_out = games_clean.copy()
games_out['appid'] = as_int(games_out['appid'])
for c in ['is_free', 'is_unreleased', 'has_relational_metadata']:
    games_out[c] = games_out[c].astype('int8')
for c in ['release_year', 'required_age', 'metacritic_score',
          'recommendations_total', 'discount_percent', 'achievement_count']:
    games_out[c] = pd.to_numeric(games_out[c], errors='coerce').astype('Int64')
for c in ['price_usd', 'price_final_usd']:
    games_out[c] = pd.to_numeric(games_out[c], errors='coerce')
games_out['release_date'] = pd.to_datetime(games_out['release_date'], errors='coerce').dt.strftime('%Y-%m-%d')
games_out = games_out[GAME_COLS].sort_values('appid').reset_index(drop=True)

export = {
    'genres':          lookup_frame(genre_lookup,    'genre_id',     'genre_name'),
    'categories':      lookup_frame(cat_lookup,      'category_id',  'category_name'),
    'developers':      lookup_frame(dev_lookup,      'developer_id', 'developer_name'),
    'publishers':      lookup_frame(pub_lookup,      'publisher_id', 'publisher_name'),
    'platforms':       lookup_frame(platform_lookup, 'platform_id',  'platform_name'),
    'games':           games_out,
    'game_genres':     link_frame(genre_links,    'genre_id'),
    'game_categories': link_frame(cat_links,      'category_id'),
    'game_developers': link_frame(dev_links,      'developer_id'),
    'game_publishers': link_frame(pub_links,      'publisher_id'),
    'game_platforms':  link_frame(platform_links, 'platform_id'),
}
for name, df in export.items():
    print(f"{name:<16} {len(df):>9,} rows")

genres                  33 rows
categories              57 rows
developers          94,983 rows
publishers          80,733 rows
platforms                3 rows
games              150,279 rows
game_genres        414,273 rows
game_categories    631,241 rows
game_developers    155,266 rows
game_publishers    147,427 rows
game_platforms     194,101 rows


### 12.2 Write the database

In [54]:
# 12.2 Write data/processed/steam_clean.sqlite
import sqlite3
from pathlib import Path

OUT_DIR = Path('../data/processed')
OUT_DIR.mkdir(parents=True, exist_ok=True)
OUT_PATH = OUT_DIR / 'steam_clean.sqlite'
OUT_PATH.unlink(missing_ok=True)   # start fresh so re-runs are idempotent

DDL = """
CREATE TABLE genres     (genre_id     INTEGER PRIMARY KEY, genre_name     TEXT NOT NULL);
CREATE TABLE categories (category_id  INTEGER PRIMARY KEY, category_name  TEXT NOT NULL);
CREATE TABLE developers (developer_id INTEGER PRIMARY KEY, developer_name TEXT NOT NULL);
CREATE TABLE publishers (publisher_id INTEGER PRIMARY KEY, publisher_name TEXT NOT NULL);
CREATE TABLE platforms  (platform_id  INTEGER PRIMARY KEY, platform_name  TEXT NOT NULL);

CREATE TABLE games (
    appid                   INTEGER PRIMARY KEY,
    name                    TEXT NOT NULL,
    is_free                 INTEGER NOT NULL CHECK (is_free IN (0, 1)),
    release_date            TEXT,
    release_year            INTEGER,
    required_age            INTEGER,
    short_description       TEXT,
    supported_languages     TEXT,
    metacritic_score        INTEGER CHECK (metacritic_score BETWEEN 0 AND 100),
    recommendations_total   INTEGER,
    discount_percent        INTEGER CHECK (discount_percent BETWEEN 0 AND 100),
    currency                TEXT,
    achievement_count       INTEGER,
    price_usd               REAL,
    price_final_usd         REAL,
    is_unreleased           INTEGER NOT NULL CHECK (is_unreleased IN (0, 1)),
    has_relational_metadata INTEGER NOT NULL CHECK (has_relational_metadata IN (0, 1)),
    pc_os_min TEXT, pc_processor_min TEXT, pc_memory_min TEXT, pc_graphics_min TEXT,
    pc_os_rec TEXT, pc_processor_rec TEXT, pc_memory_rec TEXT, pc_graphics_rec TEXT
);

CREATE TABLE game_genres (
    appid INTEGER NOT NULL REFERENCES games(appid),
    genre_id INTEGER NOT NULL REFERENCES genres(genre_id),
    PRIMARY KEY (appid, genre_id));
CREATE TABLE game_categories (
    appid INTEGER NOT NULL REFERENCES games(appid),
    category_id INTEGER NOT NULL REFERENCES categories(category_id),
    PRIMARY KEY (appid, category_id));
CREATE TABLE game_developers (
    appid INTEGER NOT NULL REFERENCES games(appid),
    developer_id INTEGER NOT NULL REFERENCES developers(developer_id),
    PRIMARY KEY (appid, developer_id));
CREATE TABLE game_publishers (
    appid INTEGER NOT NULL REFERENCES games(appid),
    publisher_id INTEGER NOT NULL REFERENCES publishers(publisher_id),
    PRIMARY KEY (appid, publisher_id));
CREATE TABLE game_platforms (
    appid INTEGER NOT NULL REFERENCES games(appid),
    platform_id INTEGER NOT NULL REFERENCES platforms(platform_id),
    PRIMARY KEY (appid, platform_id));

CREATE INDEX idx_game_genres_genre         ON game_genres(genre_id);
CREATE INDEX idx_game_categories_category  ON game_categories(category_id);
CREATE INDEX idx_game_developers_developer ON game_developers(developer_id);
CREATE INDEX idx_game_publishers_publisher ON game_publishers(publisher_id);
CREATE INDEX idx_game_platforms_platform   ON game_platforms(platform_id);
CREATE INDEX idx_games_release_year        ON games(release_year);
CREATE INDEX idx_games_is_free             ON games(is_free);
"""

con = sqlite3.connect(OUT_PATH)
con.execute("PRAGMA foreign_keys = ON")   # violations now raise during the inserts
con.executescript(DDL)

load_order = ['genres', 'categories', 'developers', 'publishers', 'platforms', 'games',
              'game_genres', 'game_categories', 'game_developers', 'game_publishers', 'game_platforms']
for name in load_order:
    export[name].to_sql(name, con, if_exists='append', index=False, chunksize=50_000)
    print(f"wrote {name:<16} {len(export[name]):>9,} rows")
con.commit()
con.close()
print(f"\nFile size: {OUT_PATH.stat().st_size / 1e6:,.0f} MB")

wrote genres                  33 rows
wrote categories              57 rows
wrote developers          94,983 rows
wrote publishers          80,733 rows
wrote platforms                3 rows
wrote games              150,279 rows
wrote game_genres        414,273 rows
wrote game_categories    631,241 rows
wrote game_developers    155,266 rows
wrote game_publishers    147,427 rows
wrote game_platforms     194,101 rows

File size: 151 MB


### 12.3 Verify by reopening the file independently

In [55]:
# 12.3 Verify the exported file (fresh connections, nothing from memory)
import duckdb

con = sqlite3.connect(OUT_PATH)
print("integrity_check:      ", con.execute("PRAGMA integrity_check").fetchone()[0])
print("foreign_key_check rows:", len(con.execute("PRAGMA foreign_key_check").fetchall()))
print("appid storage types:  ", con.execute("SELECT typeof(appid), COUNT(*) FROM games GROUP BY 1").fetchall())
con.close()

chk = duckdb.connect()
chk.execute(f"ATTACH '{OUT_PATH.as_posix()}' AS clean (TYPE sqlite, READ_ONLY)")

print("\nRow counts (file vs memory):")
for name in load_order:
    n_file = chk.execute(f"SELECT COUNT(*) FROM clean.{name}").fetchone()[0]
    print(f"  {name:<16} {n_file:>9,}   {'OK' if n_file == len(export[name]) else 'MISMATCH'}")

print("\nGames with no genre row (LEFT JOIN):",
      chk.execute("""SELECT COUNT(*) FROM clean.games g
                     LEFT JOIN clean.game_genres gg USING (appid)
                     WHERE gg.appid IS NULL""").fetchone()[0])
print("\nFlag and free split:")
print(chk.execute("""SELECT is_free, has_relational_metadata, COUNT(*) AS n
                     FROM clean.games GROUP BY 1, 2 ORDER BY 1, 2""").df().to_string(index=False))

integrity_check:       ok
foreign_key_check rows: 0
appid storage types:   [('integer', 150279)]

Row counts (file vs memory):
  genres                  33   OK
  categories              57   OK
  developers          94,983   OK
  publishers          80,733   OK
  platforms                3   OK
  games              150,279   OK
  game_genres        414,273   OK
  game_categories    631,241   OK
  game_developers    155,266   OK
  game_publishers    147,427   OK
  game_platforms     194,101   OK

Games with no genre row (LEFT JOIN): 7723

Flag and free split:
 is_free  has_relational_metadata      n
       0                        0   7265
       0                        1 124020
       1                        0      1
       1                        1  18993


### 12.4 Data dictionary (generated and checked against the file)

In [56]:
# 12.4 Data dictionary: written to docs/ and asserted against the exported schema
DOC = {
 'games': {
  'appid': 'Steam application ID. Primary key.',
  'name': 'Store listing name. Not unique; appid is the entity key.',
  'is_free': '1 if the listing is free to play, else 0. Free games have NULL prices.',
  'release_date': 'ISO date (YYYY-MM-DD). NULL when missing, out of range (before 1990 or after the scrape date) or unreleased.',
  'release_year': 'Year of release_date. NULL where release_date is NULL. 2025 is a partial year (to 2025-09-29).',
  'required_age': 'Steam age gate (0 for 99.1% of games). Weak analytical variable.',
  'short_description': 'Store-page blurb.',
  'supported_languages': 'Raw supported-language string from the store page (not parsed).',
  'metacritic_score': 'Metacritic score 0-100. NULL for 97.2% of games.',
  'recommendations_total': 'Total recommendations. NULL means not reported (reported only above about 100).',
  'discount_percent': 'Discount active at scrape time (2025-09-29), a snapshot, not a history. NULL where the listing has no price.',
  'currency': 'Original listing currency (33 codes). price_usd is populated only where currency is USD: 88,943 of 90,311 priced games. The 1,368 non-USD listings keep their currency but have NULL prices.',
  'achievement_count': 'Number of achievements. NULL where not reported.',
  'price_usd': 'Initial price in US dollars (source cents / 100). USD listings only; NULL for free or non-USD.',
  'price_final_usd': 'Price after discount in US dollars. Same coverage as price_usd.',
  'is_unreleased': '1 if the raw release date was after the scrape date (release_date is then NULL).',
  'has_relational_metadata': '0 only if the game has no genre, category, developer or publisher rows.',
  'pc_os_min': 'Minimum PC requirements, OS (raw text).',
  'pc_processor_min': 'Minimum PC requirements, processor (raw text).',
  'pc_memory_min': 'Minimum PC requirements, memory (raw text).',
  'pc_graphics_min': 'Minimum PC requirements, graphics (raw text).',
  'pc_os_rec': 'Recommended PC requirements, OS (raw text).',
  'pc_processor_rec': 'Recommended PC requirements, processor (raw text).',
  'pc_memory_rec': 'Recommended PC requirements, memory (raw text).',
  'pc_graphics_rec': 'Recommended PC requirements, graphics (raw text).'},
 'genres': {'genre_id': 'Surrogate ID (alphabetical). Primary key.',
            'genre_name': 'Canonical English genre; 154 raw translated IDs consolidated.'},
 'categories': {'category_id': 'Source category ID. Primary key.',
                'category_name': 'English store category (non-English entries removed).'},
 'developers': {'developer_id': 'Canonical developer ID. Primary key.',
                'developer_name': 'Developer name; IDs sharing a trimmed, lowercased name were merged.'},
 'publishers': {'publisher_id': 'Canonical publisher ID. Primary key.',
                'publisher_name': 'Publisher name; merged the same way as developers.'},
 'platforms': {'platform_id': '1 = windows, 2 = mac, 3 = linux. Primary key.',
               'platform_name': 'Platform name.'},
 'game_genres': {'appid': 'FK to games.', 'genre_id': 'FK to genres.'},
 'game_categories': {'appid': 'FK to games.', 'category_id': 'FK to categories.'},
 'game_developers': {'appid': 'FK to games.', 'developer_id': 'FK to developers.'},
 'game_publishers': {'appid': 'FK to games.', 'publisher_id': 'FK to publishers.'},
 'game_platforms': {'appid': 'FK to games. Source of truth for platform support.',
                    'platform_id': 'FK to platforms.'},
}
LINK_NOTE = 'Junction table; primary key is (appid, id column). One row per game-attribute pair.'

con = sqlite3.connect(OUT_PATH)
actual = {}
for (t,) in con.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name"):
    actual[t] = [(r[1], r[2]) for r in con.execute(f"PRAGMA table_info({t})")]
con.close()

assert set(actual) == set(DOC), set(actual) ^ set(DOC)
for t in actual:
    assert {c for c, _ in actual[t]} == set(DOC[t]), (t, {c for c, _ in actual[t]} ^ set(DOC[t]))

lines = ['# Data dictionary: `steam_clean.sqlite`', '',
         'Generated by `notebooks/01_data_cleaning_and_setup.ipynb` (Section 12). '
         'Scrape date: 2025-09-29. Prices are US dollars.', '']
for t in ['games', 'genres', 'categories', 'developers', 'publishers', 'platforms',
          'game_genres', 'game_categories', 'game_developers', 'game_publishers', 'game_platforms']:
    lines += [f'## {t}', '']
    if t.startswith('game_'):
        lines += [LINK_NOTE, '']
    lines += ['| Column | Type | Description |', '|---|---|---|']
    lines += [f'| `{c}` | {ty} | {DOC[t][c]} |' for c, ty in actual[t]]
    lines.append('')

Path('../docs').mkdir(exist_ok=True)
Path('../docs/data-dictionary.md').write_text('\n'.join(lines), encoding='utf-8')
print(f"Dictionary matches schema: {len(actual)} tables, {sum(len(v) for v in actual.values())} columns")

Dictionary matches schema: 11 tables, 45 columns


## 13. Summary

The raw Steam catalogue of 239,664 listings was reduced to 150,279 games and exported as a validated relational database (data/processed/steam_clean.sqlite, 11 tables, foreign keys enforced). The decisions that shape later analysis:

Entity key: appid. No name-based deduplication.
Free vs priced: is_free is the only carrier of "free"; free games have NULL prices, never 0.
Prices: converted from cents; USD listings only (no exchange-rate data). 31% of paid games have no price, concentrated in recent listings, so price analyses skew toward older listings.
Genres and categories: 154 translated genre IDs consolidated into 34 canonical genres(33 occur on game); categories filtered to English entries.
Platforms: game_platforms is the source of truth. The raw mat_supports_* flags contradicted it and were dropped.
Dates: values outside 1990-01-01..2025-09-29 set to NULL; unreleased listings flagged. 2025 is a partial year.
Developers and publishers: IDs sharing a trimmed, lowercased name were merged. Other spelling variants remain.
Coverage: 7,266 games have no genre, category, developer or publisher rows (has_relational_metadata = 0).
Sparse fields: Metacritic (97.2% null) and recommendations (86.8% null) support subsample analysis only.

### 13.1 Funnel and export summary (computed from live data)

In [57]:
# 13.1 Cleaning funnel and export summary, computed from the data, not typed in
g = games_clean
funnel = [
    ("Raw application listings",      len(applications)),
    ("Games (type == 'game')",        len(g)),
    ("Paid games",                    int((~g['is_free']).sum())),
    ("Paid games with a price",       int(g['currency'].notna().sum())),
    ("... with a USD price",          int(g['price_usd'].notna().sum())),
]
print("Cleaning funnel")
for label, n in funnel:
    print(f"  {label:<28} {n:>9,}   ({n / funnel[0][1]:.1%} of raw)")

print("\nExported tables (rows)")
for name in load_order:
    print(f"  {name:<16} {len(export[name]):>9,}")

print(f"\nGames without any relational metadata: {int((~g['has_relational_metadata']).sum()):,}")
print(f"Database file: {OUT_PATH.resolve()}  ({OUT_PATH.stat().st_size / 1e6:,.0f} MB)")

Cleaning funnel
  Raw application listings       239,664   (100.0% of raw)
  Games (type == 'game')         150,279   (62.7% of raw)
  Paid games                     131,285   (54.8% of raw)
  Paid games with a price         90,311   (37.7% of raw)
  ... with a USD price            88,943   (37.1% of raw)

Exported tables (rows)
  genres                  33
  categories              57
  developers          94,983
  publishers          80,733
  platforms                3
  games              150,279
  game_genres        414,273
  game_categories    631,241
  game_developers    155,266
  game_publishers    147,427
  game_platforms     194,101

Games without any relational metadata: 7,266
Database file: D:\Sql-eda\steam-games-sql-eda\data\processed\steam_clean.sqlite  (151 MB)
